This page walks through every public function of `simple_aws_ssm_parameter_store` with real output. It is for anyone who already uses boto3 and wants to read, write, delete, and tag SSM parameters with less error handling.

The page runs against [moto](https://docs.getmoto.org/), a local mock of AWS, so every output below was produced by actually running the code. In your own code, skip the moto cell and use a real `boto3` client.

## Setup

In [1]:
import moto

mock_aws = moto.mock_aws()
mock_aws.start()

Every function takes the boto3 SSM client you already have. This library does not create or wrap the client.

In [2]:
import boto3

ssm_client = boto3.client("ssm", region_name="us-east-1")

In [3]:
from simple_aws_ssm_parameter_store.api import ParameterType
from simple_aws_ssm_parameter_store.api import get_parameter
from simple_aws_ssm_parameter_store.api import put_parameter_if_changed
from simple_aws_ssm_parameter_store.api import delete_parameter
from simple_aws_ssm_parameter_store.api import get_parameter_tags
from simple_aws_ssm_parameter_store.api import update_parameter_tags
from simple_aws_ssm_parameter_store.api import put_parameter_tags
from simple_aws_ssm_parameter_store.api import remove_parameter_tags

## Check whether a parameter exists

In [4]:
param = get_parameter(ssm_client, "/app/database/host")
print(f"{param = }")

param = None


## Write a parameter only when its value changes

In [5]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/host",
    value="db-1.example.com",
    type=ParameterType.STRING,
    tags={"Team": "platform"},
)
print(f"{before = }")
print(f"{after.version = }")

before = None
after.version = 1


Run the same call again. The parameter already matches, so nothing is written, `after` is `None`, and the version stays at 1:

In [6]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/host",
    value="db-1.example.com",
    type=ParameterType.STRING,
)
print(f"{before.version = }")
print(f"{after = }")

before.version = 1
after = None


Change the value, and both sides are returned:

In [7]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/host",
    value="db-2.example.com",
    type=ParameterType.STRING,
)
print(f"{before.version = }, {before.value = }")
print(f"{after.version = }, {after.value = }")

before.version = 1, before.value = 'db-1.example.com'
after.version = 2, after.value = 'db-2.example.com'


Every argument you pass is compared, not only the value. A new description writes a new version, even though the value is the same:

In [8]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/host",
    value="db-2.example.com",
    description="Primary database host",
)
print(f"{after.version = }")

after.version = 3


## Read the parameter

In [9]:
param = get_parameter(ssm_client, "/app/database/host")
param.value

'db-2.example.com'

In [10]:
param.version

3

In [11]:
param.arn

'arn:aws:ssm:us-east-1:123456789012:parameter/app/database/host'

In [12]:
print(f"{param.aws_account_id = }")
print(f"{param.aws_region = }")

param.aws_account_id = '123456789012'
param.aws_region = 'us-east-1'


In [13]:
param.is_string_type

True

## SecureString

Create an encrypted parameter:

In [14]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/password",
    value="secret-1",
    type=ParameterType.SECURE_STRING,
)
print(f"{after.version = }")

after.version = 1


Without `with_decryption=True`, you get the encrypted value back. moto shows a readable placeholder here; real AWS returns base64 ciphertext.

In [15]:
get_parameter(ssm_client, "/app/database/password").value

'kms:alias/aws/ssm:secret-1'

In [16]:
get_parameter(ssm_client, "/app/database/password", with_decryption=True).value

'secret-1'

Writing the same secret again is a no-op, even without `type`: the current value is always read decrypted before the comparison.

In [17]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/password",
    value="secret-1",
)
print(f"{after = }")

after = None


## Tags

Tags are plain `dict[str, str]`. The `tags` passed when the parameter was created are there:

In [18]:
get_parameter_tags(ssm_client, "/app/database/host")

{'Team': 'platform'}

Passing `tags` again makes them the full tag set, whether or not the value changed. Tags have no versions, so `after` stays `None`:

In [19]:
before, after = put_parameter_if_changed(
    ssm_client=ssm_client,
    name="/app/database/host",
    value="db-2.example.com",
    tags={"Team": "data"},
)
print(f"{after = }")
print(get_parameter_tags(ssm_client, "/app/database/host"))

after = None
{'Team': 'data'}


In [20]:
update_parameter_tags(ssm_client, "/app/database/host", {"Env": "prod"})
get_parameter_tags(ssm_client, "/app/database/host")

{'Team': 'data', 'Env': 'prod'}

In [21]:
put_parameter_tags(ssm_client, "/app/database/host", {"Env": "prod", "Owner": "alice"})
get_parameter_tags(ssm_client, "/app/database/host")

{'Env': 'prod', 'Owner': 'alice'}

In [22]:
remove_parameter_tags(ssm_client, "/app/database/host", ["Owner"])
get_parameter_tags(ssm_client, "/app/database/host")

{'Env': 'prod'}

`put_parameter_tags` with an empty dict removes every tag:

In [23]:
put_parameter_tags(ssm_client, "/app/database/host", {})
get_parameter_tags(ssm_client, "/app/database/host")

{}

## Delete a parameter

In [24]:
delete_parameter(ssm_client, "/app/database/host")

True

and `False`, not an exception, when there was nothing to delete:

In [25]:
delete_parameter(ssm_client, "/app/database/host")

False

## Clean up

In [26]:
mock_aws.stop()